In [2]:
# First, we access Python's settings with sys.
# The time module is used to measure how long the distribution function takes to run. 
import sys, time
sys.path.append("..")   # the notebook is inside notebooks/, so go up one folder to find src.
from src.combinatorics import omega, distribution # import our functions.
# We also import the statistics module to analyze the timing data.
import statistics

# 1) We test for the largest N whose peak Omega still fits in a float.
# Since plotting needs floats, we need to know how large N can be before the peak Omega exceeds the maximum float value.
# This loop keeps repeating until something inside stops it. Each pass tries the next N.
N = 1
while True:
    try:
        float(omega(N + 1, (N + 1) // 2))
        N += 1
    except OverflowError:
        break
print("Largest N whose Omega fits in a float:", N)

# 2) We measure how long the exact distribution takes as N grows.
# We repeat each timing several times, because the time varies a little between runs, and take the average and standard deviation of the times.
repeats = 5
for N in [100, 500, 1000, 2000, 4000]: # Outer loop over the different values of N.
    times = []
    for _ in range(repeats): # Inner loop to repeat the timing for the same N multiple times.
        t0 = time.perf_counter()
        distribution(N)
        times.append(time.perf_counter() - t0)
    mean = statistics.mean(times)
    std = statistics.stdev(times)
    print(f"N = {N:5d}: {mean:.4f} ± {std:.4f} s") # We print the result.

Largest N whose Omega fits in a float: 1029
N =   100: 0.0003 ± 0.0001 s
N =   500: 0.0125 ± 0.0005 s
N =  1000: 0.0734 ± 0.0048 s
N =  2000: 0.4496 ± 0.0043 s
N =  4000: 3.2413 ± 0.0350 s


In [9]:
#Part 1a: Enumeration for N=7
import os
os.chdir("..")
from src.microstates import generate_microstates
from src.microstates import group_by_heads

N=7

microstates = list(generate_microstates(N))

print("All microstates:")
print(microstates)

# Group microstates according to the number of heads
groups = group_by_heads(microstates)

# Print the total number of microstates
print("\nTotal number of microstates:", len(microstates))

# Print microstates grouped by number of heads
print("\nMicrostates grouped by number of heads:")

for k, states in groups.items():
    print(f"{k} heads: {states}")

All microstates:
['TTTTTTT', 'TTTTTTH', 'TTTTTHT', 'TTTTTHH', 'TTTTHTT', 'TTTTHTH', 'TTTTHHT', 'TTTTHHH', 'TTTHTTT', 'TTTHTTH', 'TTTHTHT', 'TTTHTHH', 'TTTHHTT', 'TTTHHTH', 'TTTHHHT', 'TTTHHHH', 'TTHTTTT', 'TTHTTTH', 'TTHTTHT', 'TTHTTHH', 'TTHTHTT', 'TTHTHTH', 'TTHTHHT', 'TTHTHHH', 'TTHHTTT', 'TTHHTTH', 'TTHHTHT', 'TTHHTHH', 'TTHHHTT', 'TTHHHTH', 'TTHHHHT', 'TTHHHHH', 'THTTTTT', 'THTTTTH', 'THTTTHT', 'THTTTHH', 'THTTHTT', 'THTTHTH', 'THTTHHT', 'THTTHHH', 'THTHTTT', 'THTHTTH', 'THTHTHT', 'THTHTHH', 'THTHHTT', 'THTHHTH', 'THTHHHT', 'THTHHHH', 'THHTTTT', 'THHTTTH', 'THHTTHT', 'THHTTHH', 'THHTHTT', 'THHTHTH', 'THHTHHT', 'THHTHHH', 'THHHTTT', 'THHHTTH', 'THHHTHT', 'THHHTHH', 'THHHHTT', 'THHHHTH', 'THHHHHT', 'THHHHHH', 'HTTTTTT', 'HTTTTTH', 'HTTTTHT', 'HTTTTHH', 'HTTTHTT', 'HTTTHTH', 'HTTTHHT', 'HTTTHHH', 'HTTHTTT', 'HTTHTTH', 'HTTHTHT', 'HTTHTHH', 'HTTHHTT', 'HTTHHTH', 'HTTHHHT', 'HTTHHHH', 'HTHTTTT', 'HTHTTTH', 'HTHTTHT', 'HTHTTHH', 'HTHTHTT', 'HTHTHTH', 'HTHTHHT', 'HTHTHHH', 'HTHHTTT', 'HT

In [2]:
import time
import tracemalloc

def generate_microstates_binary(N):
    for state in range(1 << N):
        yield state

def benchmark_enumeration(N):
    tracemalloc.start()
    start_time = time.time()
   
    # Store the microstates in a list 
    states_list = list(generate_microstates_binary(N))
       
    elapsed_time = time.time() - start_time
    current_mem, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()
   
    total_microstates = 1 << N
    peak_mem_mb = peak_mem / (1024 * 1024)
   
    # Clear the list from memory before the next loop
    del states_list
   
    return N, total_microstates, elapsed_time, peak_mem_mb

time_limit = 300  
for n in range(20, 40):
    N, states, exec_time, mem_mb = benchmark_enumeration(n)
    print(f"N={N} | Microstates: {states} | Time: {exec_time:.4f} s | Peak Memory: {mem_mb:.6f} MB")
   
    if exec_time > time_limit:
        print(f"Limit reached at N={N}. Exceeded 5 minutes.")
        break

N=20 | Microstates: 1048576 | Time: 0.7451 s | Peak Memory: 41.961843 MB
N=21 | Microstates: 2097152 | Time: 1.5869 s | Peak Memory: 80.468733 MB
N=22 | Microstates: 4194304 | Time: 2.9823 s | Peak Memory: 161.249449 MB
N=23 | Microstates: 8388608 | Time: 5.8535 s | Peak Memory: 323.269684 MB
N=24 | Microstates: 16777216 | Time: 11.8888 s | Peak Memory: 648.236601 MB
N=25 | Microstates: 33554432 | Time: 24.0542 s | Peak Memory: 1300.052817 MB
N=26 | Microstates: 67108864 | Time: 56.8669 s | Peak Memory: 2607.500723 MB
N=27 | Microstates: 134217728 | Time: 150.2487 s | Peak Memory: 5230.130239 MB
N=28 | Microstates: 268435456 | Time: 367.8166 s | Peak Memory: 10491.068899 MB
Limit reached at N=28. Exceeded 5 minutes.
